In [ ]:
from typing import Iterable, Any
from itertools import chain as iterchain, combinations as itercomb
from collections import deque

iterflat = iterchain.from_iterable

In [ ]:
from board import DIGITS, Cell, Node, Board, Loc, Locality
from utils import countfinals, validate
from analytics import Target, MultiTarget, Link, HLink, SLink, Chain, visible_locs
from solving import orchestrator, solver, Resolution, Resolving, Resolver

In [ ]:
def fillempty(node: Node):
    if node.cell.is_empty:
        return Node(node.loc, Cell(DIGITS))
    else:
        return node

In [ ]:
async def solve_silent(initial: Board, *resolvers: Resolver):
    result = initial
    async for _, _, result in solver(initial, orchestrator(initial, *resolvers)):
        pass
    return result

In [ ]:
async def solve_logging(initial: Board, *resolvers: Resolver):
    result = initial
    async for resolver, resolution, result in solver(initial, orchestrator(initial, *resolvers)):
        print(resolver.__name__, end=": ")
        if resolution.castaways:
            print("-", " ".join(map(str, resolution.castaways)), end=" ")
        if resolution.finals:
            print("=", " ".join(map(str, resolution.finals)), end=" ")
        if resolution.highlights:
            if "zone" in resolution.highlights:
                print("@", " ".join(map(str, resolution.highlights["zone"])), end=" ")
            print("#", end=" ")
            if "anchors" in resolution.highlights:
                print(" ".join(map(str, resolution.highlights["anchors"])), end=" ")
            if "chains" in resolution.highlights:
                print(" ".join(map(str, resolution.highlights["chains"])), end=" ")
        print()
    print(validate(result))
    return result

## Basic

Singles in localities and their contraneighbours


In [ ]:
def cleanup(board: Board) -> Resolving:
    """Removing drafts contradicting with neighbouring finals"""

    for zone in Locality.all():
        neighborhood = list(board.slice(zone))
        for finode in filter(lambda n: n.cell.is_final, neighborhood):
            dig = finode.cell.final
            assert dig is not None
            contras = list(filter(lambda n: n != finode and dig in n.cell, neighborhood))
            castaways = set(Target(n.loc, dig) for n in contras)
            if castaways:
                yield Resolution(
                    castaways,
                    set(),
                    highlights={"anchors": {Target(finode.loc, dig)}, "zone": {zone}},
                )

In [ ]:
def singles(board: Board) -> Resolving:
    """Isolate singular digits in localities"""

    for zone in Locality.all():
        neighborhood = list(board.drafts(zone))
        for dig in DIGITS:
            family = list(filter(lambda n: dig in n.cell, neighborhood))
            if len(family) == 1 and len(family[0]) > 1:
                lonesome = Target(family[0].loc, dig)
                yield Resolution(
                    set(),
                    {lonesome},
                    highlights={"anchors": {lonesome}, "zone": {zone}},
                )

## Multiples

Combos of N digits

### open

Some n cells (within locality) contains only n-combo // the digits may be in other cells

=> remove the digits from all other cell of the locality

### hidden

Some n-combo contained in only n cells (within locality) // along other drafts

=> remove other drafts from the cells => it becomes open


In [ ]:
def gen_combos(m: int):
    return map(set[int], itercomb(DIGITS, m))

In [ ]:
def openmults(board: Board, mult: int) -> Resolving:
    """Clar out spoiling neighbours of open multiples in each zone"""
    for zone in Locality.all():
        neighborhood = tuple(board.drafts(zone))
        for combo in gen_combos(mult):
            habitat = set(filter(lambda n: combo <= n.cell, neighborhood))  # all nodes containing the combo
            spoilers = tuple(filter(lambda n: n not in habitat and combo <= n.cell, neighborhood))  # all neighbors with the combo digits
            if len(spoilers) > 0:
                castaways = set(Target(n.loc, d) for n in spoilers for d in n.cell & combo)
                anchors = set(MultiTarget(n.loc, n.cell & combo) for n in habitat)
                yield Resolution(
                    castaways,
                    set(),
                    highlights={"anchors": anchors, "zone": {zone}},
                )


def openmults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return openmults(board, mult)

    resolver.__name__ = f"openmults[{mult}]"
    return resolver

In [ ]:
def unhidemults(board: Board, mult: int) -> Resolving:
    """Clean up cellmates of hidden multiples"""
    for zone in Locality.all():
        neighborhood = tuple(board.drafts(zone))
        for combo in gen_combos(mult):
            habitat = set(filter(lambda n: n.cell & combo, neighborhood))  # all nodes containing the combo
            habitants = set(iterflat(n.cell & combo for n in habitat))  # all combo digits in the nodes
            if len(habitat) == mult and len(habitants) == mult:
                spoilers = set(filter(lambda n: n.cell - combo, habitat))  # cells with other digits
                if len(spoilers):
                    castaways = set(Target(n.loc, d) for n in spoilers for d in n.cell - combo)  # the other digits targeted
                    anchors = set(MultiTarget(n.loc, n.cell & combo) for n in habitat)  # the combo digits anchored
                    yield Resolution(
                        castaways,
                        set(),
                        highlights={"anchors": anchors, "zone": {zone}},
                    )


def unhidemults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return unhidemults(board, mult)

    resolver.__name__ = f"unhidemults[{mult}]"
    return resolver

## Links

### hard links

Represent XOR relation

Criteria:

- only 2 drafts of same digit in a locality
- only 2 drafts in a cell

### soft links

Represent NAND relation

Criteria:

- any 2 drafts of same digit in a locality
- any 2 drafts in a cell

The criteria are totally independent of board content (assuming target digits exist)

### chains

Alterating link chains: (-xor-nand-)^n

(A-xor-B-nand-)^n-xor-D and (A-nand-D) => (A-xor-D)

All {x: (x-nand-A) and (x-nand-D)} can be eliminated


In [ ]:
def scan_hard(board: Board) -> Iterable[tuple[Target, Target]]:
    """Scan all xor pairs on the board"""

    def scan_cell(loc: Loc):
        node = board.get(loc)
        if len(node.cell) == 2:
            d1, d2 = node.cell
            yield (Target(node.loc, d1), Target(node.loc, d2))

    def scan_locality(zone: Locality):
        for d in DIGITS:
            family = tuple(board.drafts(zone, lambda n: d in n.cell))
            if len(family) == 2:
                n1, n2 = family
                yield (Target(n1.loc, d), Target(n2.loc, d))

    for node in board.drafts():
        yield from scan_cell(node.loc)

    for zone in Locality.all():
        yield from scan_locality(zone)


def search_links(board: Board) -> Iterable[HLink]:
    return (HLink(targets) for targets in scan_hard(board))

In [ ]:
# def check_linksoft(lnk1: Link, t2: Target):
#     return check_soft(lnk1[0], t2) and check_soft(lnk1[1], t2)

In [ ]:
def search_chains(links: Iterable[HLink], max_length: int) -> Iterable[Chain]:
    """Search for all chains"""
    # breadth-first graph search
    frontier = deque[Chain](Chain.init(lnk) for lnk in links)  # queue
    explored = set[Chain]()
    while frontier:
        chain = frontier.popleft()
        if len(chain) > 1:
            yield chain  # yielding all found chains
        explored.add(chain)
        if len(chain) >= max_length:
            continue
        frontier.extend(ext for ext in expand_alc(chain, links) if ext not in explored and ext not in frontier)


def expand_alc(chain: Chain, links: Iterable[HLink]) -> Iterable[Chain]:
    """Expand chain to one of other hard links in the pool"""
    e1, e2 = chain.edges

    # closing loop
    if len(chain) > 2 and isinstance(chain[0], HLink) and isinstance(chain[-1], HLink) and Target.check_nand(e1, e2):
        yield Chain.extend(chain, SLink((e2, e1)))

    anchors = chain.anchors()

    # iter through other links not pointing back to the chain
    for link in filter(lambda l: l[0] not in anchors and l[1] not in anchors, links):
        x1, x2 = link
        if Target.check_nand(e2, x1):
            yield Chain.extend(chain, SLink((e2, x1)), link)
        if Target.check_nand(e2, x2):
            yield Chain.extend(chain, SLink((e2, x2)), link.reversed())

In [ ]:
def match_loop(chain: Chain):
    """ALC loop"""
    return len(chain) > 2 and len(chain) % 2 == 0 and chain.is_loop


def match_rope(chain: Chain):
    """ALC with matching edges"""
    e1, e2 = chain.edges
    return len(chain) > 2 and len(chain) % 2 == 1 and e1.dig == e2.dig

In [ ]:
def iter_visible(board: Board, a1: Target, a2: Target, anchors: set[Target]) -> Iterable[Target]:
    for node in board.drafts(visible_locs(a1.loc, a2.loc)):
        for trg in Target.iter_node(node):
            if trg not in anchors and Target.check_nand(a1, trg) and Target.check_nand(a2, trg):
                yield trg


def resolve_loop(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible for each soft link"""
    anchors = chain.anchors()

    for link in filter(lambda lnk: isinstance(lnk, SLink), chain):
        t1, t2 = link
        spoilers = set(iter_visible(board, t1, t2, anchors))
        if len(spoilers):
            yield Resolution(
                spoilers,
                set(),
                highlights={"anchors": {t1, t2}, "links": set(chain)},
            )


def resolve_rope(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible from both edges"""
    e1, e2 = chain.edges
    anchors = chain.anchors()

    spoilers = set(iter_visible(board, e1, e2, anchors))
    if len(spoilers):
        yield Resolution(
            spoilers,
            set(),
            highlights={"anchors": {e1, e2}, "links": set(chain)},
        )


def resolve_chains(current: Board, max_length: int) -> Resolving:
    links = set(search_links(current))
    for chain in search_chains(links, max_length):
        if match_loop(chain):
            yield from resolve_loop(current, chain)
        elif match_rope(chain):
            yield from resolve_rope(current, chain)


def chains_(max_length: int):
    def resolver(board: Board) -> Resolving:
        return resolve_chains(board, max_length)

    resolver.__name__ = f"chains[max={max_length}]"
    return resolver

## A puzzle


In [ ]:
from utils import parse

# simple: solvable by basics + multiples[2]
# puzzle = parse("""
# ....8.41.
# 6........
# .29..58..
# 8...7.2..
# .........
# .7......5
# 2...3...8
# ...5...3.
# .4.7.9..6
# """)

# expert
puzzle = parse("""
.........
..9....4.
.13.75...
..7..1.36
.6.29...7
8....6...
.......13
...86..95
2........
""")

# extreme level: not solvable by myself
# puzzle = parse("""
# .8.....52
# .......87
# ....98...
# 4...3.6..
# .2.7.....
# .........
# 6..8.2...
# ...5.91..
# 9........
# """)


puzzle = Board.transform(puzzle, fillempty)

### UI

async ui with inspection of resolutions


In [ ]:
# widgets

from collections import Counter
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from canvas import SudokuCanvas


def format_options(objects: Iterable[Any]):
    return tuple((str(obj), obj) for obj in objects)


def on_change(widget: w.Widget):
    def wrapper(handler):
        return widget.observe(handler, "value")

    return wrapper


debug_view = w.Output()

LINKSTYLES = {"HLink": "HARD", "SLink": "SOFT"}
column_layout = w.Layout(width="auto", height="100%", flex_flow="column", align_items="stretch")


canvas = SudokuCanvas()


def deselect(widget):
    widget.value = () if isinstance(widget, w.SelectMultiple) else None


@canvas.on_client_ready
def init_canvas():
    canvas[2].global_alpha = 0.5
    canvas.draw_grid()


@canvas.on_mouse_up
def on_canvas_click(x, y):
    # targ = canvas.map_target(x, y)
    # TODO: highlight/dehighlight, search links
    pass


def highlight_targets(targets: Iterable[Target] | Iterable[MultiTarget], color: str):
    for trg in targets:
        if isinstance(trg, Target):
            canvas.highlight_segment(trg.loc, trg.dig, color=color)
        elif isinstance(trg, MultiTarget):
            canvas.highlight_segments(trg.loc, trg.digs, color=color)


def highlight_links(links: Iterable[Link], color: str):
    for lnk in links:
        t1, t2 = lnk
        canvas.highlight_link(t1.loc, t1.dig, t2.loc, t2.dig, style=LINKSTYLES[lnk.__class__.__name__], color=color)


def wCounterText(label: str):
    return w.Text(
        label,
        layout=dict(width="auto"),
    )


counters_labels = {dig: wCounterText("...") for dig in DIGITS}
countotal_label = wCounterText("...")


def show_counters(counters: Counter):
    for dig, cnt in counters.items():
        counters_labels[dig].value = f"({dig}): {cnt}"
        counters_labels[dig].style.background = "var(--jp-success-color2)" if cnt == 9 else ""
    total = counters.total()
    countotal_label.value = f"Total: {total}"
    countotal_label.style.background = "var(--jp-success-color1)" if total == 81 else ""


status_label = w.Text(layout=dict(width="auto"), style=dict(text_color="white"))


def show_status(status: str):
    status_label.value = status
    status_label.style.visibility = "visible"
    if status == "SOLVED":
        status_label.style.background = "var(--jp-success-color0)"
    elif status == "BROKEN":
        status_label.style.background = "var(--jp-error-color0)"
    else:
        status_label.style.background = "var(--jp-info-color3)"


resolver_label = w.Label(value="")
resolver_count = w.Label(value="")

btn_running = w.Button(icon="gear spin", button_style="warning", style=dict(font_size="large"))
btn_running.layout.visibility = "hidden"
btn_continue = w.Button(description="Continue", disabled=True, button_style="primary")
btn_continue.layout.visibility = "hidden"

select_inspecting = w.SelectMultiple(
    options=[],
    description="Inspecting:",
    value=[],
    layout=dict(flex_flow="column", width="auto", align_items="flex-start"),
    indent=False,
    style=dict(description_width="auto", text_align="left"),
    rows=20,
)
select_inspecting.layout.visibility = "hidden"


column_layout = w.Layout(
    width="auto",
    height="auto",
    flex_flow="column",
    justify_content="flex-start",
    align_items="stretch",
    margin="0 4px",
    overflow="hidden",
)

selecting_anchors = w.SelectMultiple(
    description="Anchors",
    options=[],
    value=[],
    layout=column_layout,
    indent=False,
    style=dict(description_width="auto"),
)

selecting_links = w.SelectMultiple(
    description="Links",
    options=[],
    value=[],
    layout=column_layout,
    indent=False,
    style=dict(description_width="auto"),
)

selecting_chains = w.SelectMultiple(
    description="Chains",
    options=[],
    value=[],
    layout=column_layout,
    indent=False,
    style=dict(description_width="auto"),
)


@on_change(selecting_anchors)
def on_selecting_anchors(change):
    if change.old and not canvas.caching:
        canvas.clear_highlights()
    if change.new:
        selected: list[Target] = list(change.new)
        with hold_canvas():
            highlight_targets(selected, "blue")
            selecting_links.value = []
            selecting_chains.value = []


@on_change(selecting_links)
@debug_view.capture()
def on_selecting_links(change):
    if change.old and not canvas.caching:
        canvas.clear_highlights()
    if change.new:
        selected: list[Link] = list(change.new)
        # print("links selected", list(map(str, selected)))
        with hold_canvas():
            highlight_links(selected, "blue")
            selecting_anchors.value = []
            selecting_chains.value = []


@on_change(selecting_chains)
@debug_view.capture()
def on_selecting_chains(change):
    if change.old and not canvas.caching:
        canvas.clear_highlights()
    if change.new:
        selected: list[Chain] = list(change.new)
        # print("chains selected", list(map(str, selected)))
        with hold_canvas():
            highlight_links(iterflat(selected), color="blue")
            selecting_anchors.value = []
            selecting_links.value = []


In [ ]:
# public methods kinda


def draw_board(board: Board):
    canvas.draw_board(board)


def show_targets(targets: Iterable[Target]):
    highlight_targets(list(targets), "orange")


def show_finals(targets: Iterable[Target]):
    with hold_canvas():
        for trg in targets:
            canvas.highlight_final(trg, color="purple")


def show_anchors(anchors: Iterable[Target] | set[MultiTarget], select: bool = False):
    selecting_anchors.options = sorted(format_options(anchors), key=lambda opt: opt[0])
    if select:
        selecting_anchors.value = list(anchors)


def show_links(links: Iterable[Link], select: bool = False):
    selecting_links.options = sorted(format_options(links), key=lambda opt: opt[0])
    if select:
        selecting_anchors.value = []
        selecting_links.value = list(links)


def show_chains(chains: Iterable[Chain], select: bool = False):
    selecting_chains.options = format_options(chains)
    if select:
        selecting_anchors.value = []
        selecting_links.value = []
        selecting_chains.value = list(chains)


btn_reload = w.Button(description="Reload")


@btn_reload.on_click
def on_reload(btn):
    canvas.clear_highlights()
    draw_board(puzzle)
    show_counters(countfinals(puzzle))
    show_status(validate(puzzle))

In [ ]:
display(
    w.HBox(
        [
            w.VBox(
                [w.Label("Status"), *counters_labels.values(), countotal_label, status_label],
                layout=dict(align_items="stretch", width="6em"),
            ),
            canvas,
            w.VBox([
                btn_reload,
                w.HBox([resolver_label, resolver_count]),
                btn_running,
                btn_continue,
                select_inspecting,
            ]),
            selecting_anchors,
            selecting_links,
            selecting_chains,
        ],
        layout=dict(justify_content="flex-start", align_items="stretch"),
    )
)

In [ ]:
debug_view

In [ ]:
import asyncio


def wait_continue():
    btn_continue.disabled = False
    future = asyncio.Future()

    def on_click(b):
        btn_continue.on_click(on_click, remove=True)
        btn_continue.disabled = True
        future.set_result(True)

    btn_continue.on_click(on_click)
    return future


def render_resolution(res: Resolution):
    print(res)
    with hold_canvas():
        show_targets(res.castaways)
        show_finals(res.finals)
        if res.highlights is not None:
            if "links" in res.highlights:
                show_links(res.highlights["links"], select=True)
            if "chains" in res.highlights:
                show_chains(res.highlights["chains"], select=True)
            if "anchors" in res.highlights:
                show_anchors(res.highlights["anchors"], select=True)


def clear_resolution():
    canvas.clear_highlights()
    show_targets({})
    show_anchors({})
    show_links({})


def render_result(result: Board):
    with hold_canvas():
        draw_board(result)
        show_counters(countfinals(result))


puzzle: Board


async def solve_ui(*resolvers: Resolver):
    global puzzle

    select_inspecting.options = [r.__name__ for r in resolvers]
    select_inspecting.value = select_inspecting.options[:]
    select_inspecting.layout.visibility = "visible"

    btn_continue.layout.visibility = "visible"
    btn_continue.disabled = True

    render_result(puzzle)
    async for resolver, resolution, result in solver(puzzle, orchestrator(puzzle, *resolvers)):
        resolver_label.value = resolver.__name__
        resolver_count.value = f"-{len(resolution.castaways)} ={len(resolution.finals)}"
        btn_running.layout.visibility = "hidden"
        if resolver.__name__ in select_inspecting.value:
            btn_continue.disabled = False
            render_resolution(resolution)
            await wait_continue()
            clear_resolution()
            btn_continue.disabled = True
            await asyncio.sleep(0.2)
        puzzle = result
        render_result(puzzle)
        btn_running.layout.visibility = "visible"

    resolver_label.value = ""
    resolver_count.value = ""

    btn_running.layout.visibility = "hidden"
    btn_continue.layout.visibility = "hidden"
    select_inspecting.layout.visibility = "hidden"
    select_inspecting.options = []
    show_status(validate(puzzle))

In [ ]:
puzzle = await solve_silent(puzzle, cleanup, singles)
# puzzle = await solve_silent(puzzle, openmults_(2), unhidemults_(2), openmults_(3), unhidemults_(3))
draw_board(puzzle)

In [ ]:
task = asyncio.create_task(
    solve_ui(
        cleanup,
        singles,
        openmults_(2),
        unhidemults_(2),
        openmults_(3),
        unhidemults_(3),
        openmults_(4),
        unhidemults_(4),
        openmults_(5),
        unhidemults_(5),
        chains_(6),
    )
)

In [ ]:
task.cancel()